# Media Source Extraction Pipeline

Research question: does the variety of news sources a person uses relate to more positive attitudes toward other races?

To measure variety, we need to pull out and clean up every media source survey respondents mentioned. People write freely — abbreviations, typos, full names, URLs, whole sentences — so this takes several cleaning steps.

Pipeline:
```
Raw survey cells
    → Step 1: Split cells into individual source names
    → Step 2: Clean each string (lowercase, strip URLs, etc.) + subcategorize by survey column
    → Step 3: Frequency check — spot top sources and obvious duplicates
    → Step 4: Pull known sources out of long sentences
    → Step 5: Handle 'like', 'from', and similar patterns
    → Step 6: Map all variants to one standard name per source
    → Step 7: Label each source as Mainstream Media / Social Media / Non-News
    → Step 7b: Political leaning from Media Bias Fact Check
    → Step 8: Audit table — all unique sources with category, unified name, frequency
    → Step 9: Per-person counts by source type and political leaning
    → Step 10: Export
```

Priority: get the common sources right. Sources mentioned by only 1–2 people can be imprecise.

# Setup and data load

In [1]:
import pandas as pd
import re
from collections import Counter
from openpyxl.styles import Alignment
from openpyxl.utils import get_column_letter

df = pd.read_csv('media_sources_internship.csv')
print(df.shape)
print(list(df.columns))

(928, 21)
['PROLIFIC_PID', 'Facebook', 'Facebook_freq', 'Instagram', 'Instagram_freq', 'TV news programs', 'TV news programs_freq', 'TikTok', 'TikTok_freq', 'Twitter/X', 'Twitter/X_freq', 'YouTube', 'YouTube_freq', 'online news sites', 'online news sites_freq', 'podcasts', 'podcasts_freq', 'printed newspapers', 'printed newspapers_freq', 'the radio', 'the radio_freq']


In [2]:
# Columns where the survey platform was social media (Facebook, Instagram, etc.)
# If someone wrote 'CNN' in the YouTube column, that's MM accessed via social media
SM_SURVEY_COLS = ['Facebook', 'Instagram', 'TikTok', 'Twitter/X', 'YouTube']

# All other text columns are traditional media contexts
MM_SURVEY_COLS = ['TV news programs', 'online news sites', 'podcasts',
                  'printed newspapers', 'the radio']

# All source text columns together (drop ID and pre-counted freq columns)
source_cols = [
    col for col in df.columns
    if not col.endswith('_freq')
    and col != 'PROLIFIC_PID'
]

# Stack all source columns into one long list, but keep track of which column each came from
# We need the column name later to decide MM-in-SM subcategory
all_text_with_col = (
    df[source_cols]
    .stack()
    .reset_index()
    .rename(columns={'level_1': 'survey_col', 0: 'text'})
    .assign(text=lambda d: d['text'].astype(str))
)

all_text = all_text_with_col['text']
print(f"Total cells: {len(all_text)}")
all_text.sample(10, random_state=42).tolist()

Total cells: 2771


['Fox News, Google News, Apple News, CNN',
 'New York Times, cnn, Washington post, tmz, nbc',
 'CNN; MSNBC; NBC',
 'Fox; Andrew Napalitano; Tucker Carlson',
 'Opera News',
 'cnn;bbc;skynews',
 'CNN; Fox; BBC',
 'Money GPS, Wall Street Journal, Bloomburge',
 'Last week tonight; daily show',
 'Washington Post; Politico']

# Step 1: Split cells into individual source names

Each cell can have multiple sources separated by commas, semicolons, newlines, or 'and'.
The tricky case is 'and' — some source names contain it (e.g. 'Aba and Preach').
Fix: only split on 'and' when the text before it is 30 characters or less (likely a short name, not a sentence).

Sources that contain 'and' in their name — add here if you find more:

In [3]:
PROTECTED_NAMES = {
    'aba and preach',
    'faith and freedom coalition',
    'guns and butter',
    'law and crime',
    'crime and justice',
    'meet the press and msnbc',
    'breaking points with krystal and saagar',
    'good and evil',
}

def split_sources(text):
    # Split on unambiguous delimiters — never part of a source name
    # ; , newline : / and ellipsis
    hard_delims = r'[;,\n:/]|\.\.\.+'
    segments = re.split(hard_delims, text)

    result = []
    for seg in segments:
        # Protected names stay whole — never split them on 'and'
        if seg.strip().lower() in PROTECTED_NAMES:
            result.append(seg)
            continue

        parts = re.split(r'\s+and\s+', seg, flags=re.IGNORECASE)

        if len(parts) == 1:
            result.append(seg)
        else:
            current = parts[0]
            for next_part in parts[1:]:
                if len(current.strip()) <= 30:
                    # Short left side — 'and' is a list separator, e.g. 'CNN and BBC' → ['CNN', 'BBC']
                    result.append(current)
                    current = next_part
                else:
                    # Long left side — 'and' is part of the name or sentence, keep together
                    current = current + ' and ' + next_part
            result.append(current)

    return [s.strip() for s in result if s.strip()]


# Quick tests
test_cases = [
    ("CNN, BBC and Fox",            "expect 3 items"),
    ("NYT; Washington Post; NPR",   "expect 3 items"),
    ("Aba and Preach, NYT",         "expect 2 items — Aba and Preach stays whole"),
    ("cnn, aba and preach and nyt", "expect cnn / aba and preach / nyt"),
    ("CNN\nBBC\nFox",               "expect 3 items from newlines"),
    ("online news articles from cbs", "expect 1 item — stays long for step 5"),
]

print("=== Split tests ===")
for text, note in test_cases:
    result = split_sources(text)
    print(f"  Input : {repr(text)}")
    print(f"  Note  : {note}")
    print(f"  Output: {result}")
    print()

=== Split tests ===
  Input : 'CNN, BBC and Fox'
  Note  : expect 3 items
  Output: ['CNN', 'BBC', 'Fox']

  Input : 'NYT; Washington Post; NPR'
  Note  : expect 3 items
  Output: ['NYT', 'Washington Post', 'NPR']

  Input : 'Aba and Preach, NYT'
  Note  : expect 2 items — Aba and Preach stays whole
  Output: ['Aba and Preach', 'NYT']

  Input : 'cnn, aba and preach and nyt'
  Note  : expect cnn / aba and preach / nyt
  Output: ['cnn', 'aba', 'preach', 'nyt']

  Input : 'CNN\nBBC\nFox'
  Note  : expect 3 items from newlines
  Output: ['CNN', 'BBC', 'Fox']

  Input : 'online news articles from cbs'
  Note  : expect 1 item — stays long for step 5
  Output: ['online news articles from cbs']



# Step 2: Clean each candidate string + note which survey column it came from

Cleaning: lowercase, strip URLs, remove domain endings, remove noise punctuation.

We also record a subcategory here:
- MM-in-SM = a news source written in a social media column (e.g. someone wrote 'CNN' in the YouTube column)
- MM = a news source written in a traditional media column
This subcategory is stored alongside the source and used in Step 9.

In [4]:
def clean(s):
    s = s.lower()

    # Remove URL prefixes people paste in: 'https://', 'http://', 'www.', '//'
    s = re.sub(r'https?://|www\.|//', '', s)

    # Remove domain endings — '.co.uk' must come before '.co'
    # e.g. 'cnn.com' → 'cnn'   'bbc.co.uk' → 'bbc'
    s = re.sub(r'\.co\.uk|\.com|\.org|\.net|\.co|\.uk|\.gov|\.edu', '', s)

    # Underscores to spaces (common in Twitter/Reddit usernames)
    s = s.replace('_', ' ')

    # Remove symbols never in source names: @ ™ quotes brackets
    s = re.sub(r'[@™\'\"()\[\]]', '', s)

    # Drop non-ASCII (encoding bugs like â€™)
    s = s.encode('ascii', 'ignore').decode('ascii')

    # Strip leftover slashes, hyphens, dots at start/end
    # e.g. '/cnn' → 'cnn'   '.fox' → 'fox'
    s = s.strip(' /-.')

    return s


# Explode each row in all_text_with_col into one row per split piece,
# keeping the survey_col so we know where it came from
rows = []
for _, row in all_text_with_col.iterrows():
    pieces = split_sources(row['text'])
    for piece in pieces:
        cleaned = clean(piece)
        if cleaned:
            rows.append({'cleaned': cleaned, 'survey_col': row['survey_col']})

exploded_df = pd.DataFrame(rows)

# Add subcategory column:
# MM-in-SM = news source found in a social media survey column
# MM = news source found in a traditional media survey column
# (SM and NN will be assigned properly in Step 7 — this is just origin tracking)
exploded_df['origin'] = exploded_df['survey_col'].apply(
    lambda col: 'SM_col' if col in SM_SURVEY_COLS else 'MM_col'
)

all_cleaned = exploded_df['cleaned']

print(f"Total source mentions (with repeats): {len(all_cleaned)}")
print(f"Unique strings after cleaning: {all_cleaned.nunique()}")
print()
all_cleaned.sample(20, random_state=42).tolist()

Total source mentions (with repeats): 6355
Unique strings after cleaning: 2506



['univision',
 'cbs news',
 'huffington post',
 'msnbc',
 'democracy now',
 'cnn',
 'various pages that pop up',
 'firstpost',
 'cnn',
 'washington post',
 'epoch times',
 'abc',
 'reuters',
 'feautured',
 'various podcasts',
 'npr',
 'abc',
 'reuters',
 'washington post',
 'reuters']

In [5]:
# Frequency table after cleaning — sources mentioned 3 or more times
freq_after_clean = (
    all_cleaned
    .value_counts()
    .reset_index()
)
freq_after_clean.columns = ['source', 'count']
freq_after_clean['cumulative_pct'] = (
    freq_after_clean['count'].cumsum() / freq_after_clean['count'].sum() * 100
).round(1)

top_sources = freq_after_clean[freq_after_clean['count'] >= 3].copy()

print(f"Unique strings mentioned 3+ times: {len(top_sources)}")
print(f"These cover {top_sources['count'].sum()} of {len(all_cleaned)} total mentions")
print()
with pd.option_context('display.max_rows', 300, 'display.max_colwidth', 60):
    print(top_sources.to_string())

Unique strings mentioned 3+ times: 250
These cover 3896 of 6355 total mentions

                      source  count  cumulative_pct
0                        cnn    623             9.8
1                   fox news    188            12.8
2                      msnbc    184            15.7
3                        npr    157            18.1
4             new york times    139            20.3
5                        bbc    130            22.4
6                        nbc    114            24.2
7                        abc    106            25.8
8            washington post     89            27.2
9                        fox     89            28.6
10                       cbs     81            29.9
11                   reuters     64            30.9
12                  abc news     52            31.7
13               google news     45            32.4
14                      cnbc     44            33.1
15        the new york times     44            33.8
16       wall street journal     44 

# Step 3: Coverage check

How many unique strings do we need to cover 80%, 90%, 95% of all mentions?
This tells us how much effort to put into Step 6.

In [6]:
thresholds = [80, 90, 95]
print("Coverage:")
for t in thresholds:
    n = (freq_after_clean['cumulative_pct'] <= t).sum()
    print(f"  Top {n} unique strings cover {t}% of all mentions")

print(f"\nTotal unique strings: {len(freq_after_clean)}")

Coverage:
  Top 1238 unique strings cover 80% of all mentions
  Top 1873 unique strings cover 90% of all mentions
  Top 2191 unique strings cover 95% of all mentions

Total unique strings: 2506


# Step 4: Pull known sources out of long sentences

Some respondents write whole sentences: 'I mainly watch CNN for political news'.
These will never match 'cnn' exactly.

Fix: build a list of all short cleaned strings (30 chars or less) that are not generic junk words.
For any string 25+ characters long, scan it for one of these known short names.
If found, replace the whole sentence with just the source name.

Known names are sorted longest-first so 'new york times' is found before just 'times'.

In [7]:
# Generic words that happen to be short but are not source names
# One per line for easy editing
JUNK_WORDS = {
    'actors',
    'any',
    'app',
    'apps',
    'clips',
    'content',
    'creators',
    'daily',
    'entertainment',
    'etc',
    'everything',
    'family',
    'friends',
    'games',
    'generally',
    'just',
    'local',
    'mainly',
    'many',
    'media',
    'more',
    'mostly',
    'movie',
    'movies',
    'multiple',
    'music',
    'n/a',
    'na',
    'news',
    'no',
    'none',
    'not',
    'nothing',
    'often',
    'online',
    'other',
    'others',
    'people',
    'phone',
    'posts',
    'radio',
    'shows',
    'some',
    'sometimes',
    'sports',
    'stuff',
    'television',
    'there',
    'things',
    'today',
    'tv',
    'usually',
    'users',
    'various',
    'videos',
    'website',
    'websites',
    'whatever',
    'yes',
}

unique_cleaned = all_cleaned.unique().tolist()

known_names = sorted(
    [s for s in unique_cleaned if len(s) <= 30 and s not in JUNK_WORDS],
    key=len,
    reverse=True   # longest first — 'new york times' matched before 'times'
)

print(f"Known source names: {len(known_names)} entries")
print(f"Longest 20: {known_names[:20]}")

Known source names: 2216 entries
Longest 20: ['entertainment news from actors', 'the beverly hills courier post', 'so i can find everything there', 'airone radio  general stations', 'i pay attention to msnbc clips', 'jewish voice for peace jvplive', 'cnbc mainly for financial news', 'local radio station talk shows', 'i watch local news programming', 'delaware state police newsroom', 'friends out local news sources', 'others posting new on the page', 'whatever is on my for you page', 'usually news channels like fox', 'not following any general feed', 'online news articles from cbs', 'local news. multiple channels', 'black information network bin', 'i dont follow specifc sources', 'african diaspora news channel']


In [8]:
def pull_source_from_sentence(s, known_names):
    # Already short — leave it
    if len(s) < 25:
        return s

    # Scan for any known source name as a whole word inside this sentence
    # \b = word boundary — stops 'ox' matching inside 'fox'
    for name in known_names:
        if re.search(r'\b' + re.escape(name) + r'\b', s):
            return name

    return s


all_anchored = all_cleaned.apply(lambda s: pull_source_from_sentence(s, known_names))

long_mask = all_cleaned.str.len() >= 25
resolved = (all_anchored[long_mask] != all_cleaned[long_mask]).sum()
unresolved = long_mask.sum() - resolved

print(f"Long strings (25+ chars): {long_mask.sum()}")
print(f"  Resolved: {resolved}")
print(f"  Still unresolved: {unresolved}")
print()

examples = pd.DataFrame({
    'Before': all_cleaned[long_mask].values,
    'After':  all_anchored[long_mask].values
})
examples = examples[examples['Before'] != examples['After']]
print("=== Sample resolved ===")
with pd.option_context('display.max_rows', 20, 'display.max_colwidth', 80):
    print(examples.head(20).to_string())

Long strings (25+ chars): 387
  Resolved: 175
  Still unresolved: 212

=== Sample resolved ===
                                                                                                        Before                  After
2                                                                   content creators that i follow like arnold       content creators
3       i check out some of what is trending and get entertainment news from figures that i follow like arnold               trending
4                                                                      political news from the majority report    the majority report
6                                          entertainment news from various content creators like alanah pearce       content creators
9                                                          other major news outlets usually google recommended                 google
11                     whatever credible news sources i can find when backing up information i found 

# Step 5: Handle 'like', 'from', and similar patterns

Some sentences need special extraction:
- 'content creators like Alanah Pearce' → extract after 'like'
- 'online news articles from CBS' → extract after 'from'
- 'ohio station channel 3 (nbc)' → pull the known name from anywhere in the string
- 'npr (national public radio)' → 'npr national public radio' → unified to NPR in Step 6

The 'from' pattern catches cases like:
- 'online news articles from cbs' → 'cbs'
- 'various pages that report news from cnn' → 'cnn'
- 'get my news from the new york times' → 'new york times'

In [9]:
def extract_keyword_pattern(s):
    # Only apply to strings still long after step 4
    if len(s) < 25:
        return s

    # Pattern: extract text after 'like' — e.g. 'content creators like Alanah Pearce'
    match = re.search(r'\blike\s+([a-z][^,;:\n/]{1,40})', s)
    if match:
        candidate = match.group(1).strip()
        if len(candidate.split()) <= 5 and candidate not in JUNK_WORDS:
            return candidate

    # Pattern: extract text after 'from' — e.g. 'online news articles from cbs'
    match = re.search(r'\bfrom\s+([a-z][^,;:\n/]{1,40})', s)
    if match:
        candidate = match.group(1).strip()
        # Only keep if it's short and not a noise phrase
        if len(candidate.split()) <= 5 and candidate not in JUNK_WORDS:
            # Extra check: skip phrases like 'from friends', 'from people i follow'
            noise = ['friends', 'people', 'everyone', 'my', 'the home', 'there',
                     'elsewhere', 'different', 'various', 'all over', 'both sides']
            if not any(n in candidate for n in noise):
                return candidate

    return s


still_long = all_anchored.str.len() >= 25
all_extracted = all_anchored.copy()
all_extracted[still_long] = all_anchored[still_long].apply(extract_keyword_pattern)

pattern_resolved = (all_extracted[still_long] != all_anchored[still_long])
print(f"Strings resolved by 'like'/'from' patterns: {pattern_resolved.sum()}")

if pattern_resolved.sum() > 0:
    pattern_examples = pd.DataFrame({
        'Before': all_anchored[still_long][pattern_resolved].values,
        'After':  all_extracted[still_long][pattern_resolved].values
    })
    with pd.option_context('display.max_rows', 20, 'display.max_colwidth', 80):
        print(pattern_examples.head(20).to_string())

Strings resolved by 'like'/'from' patterns: 6
                                                                          Before             After
0                                   show pages that i follow like the black keys    the black keys
1                                               entertainment news like movieweb          movieweb
2                                                streaming from varoius channels  varoius channels
3                                                  online news articles from cbs               cbs
4  im counting this because of announcements that i see linked from other places      other places
5                                                 usually news channels like fox               fox


In [10]:
# What is still unresolved — will be labeled Non-News in Step 7
# Note: 'npr (national public radio)' type strings are handled in Step 6 via alias map
still_unresolved = all_extracted[all_extracted.str.len() >= 25]
print(f"Still unresolved (25+ chars): {len(still_unresolved)}")
print()
with pd.option_context('display.max_rows', 50, 'display.max_colwidth', 80):
    print(still_unresolved.value_counts().head(50).reset_index().to_string())

Still unresolved (25+ chars): 206

                                                                                                       cleaned  count
0                                                                                    npr national public radio      2
1                                                                                 atlanta journal constitution      2
2                                                                                only what shows up in my feed      2
3                                                                          just following links from elsewhere      2
4   some advocates that i follow regarding the social issues i care about. they repost reels and news articles      1
5                         just tend to view whatever is appearing on the timeline and is part of relevant news      1
6                                                                           various clips no specific channels      1
7                    

# Step 6: Map all variants to one standard name

Why not fuzzy matching? Short abbreviations break it badly:
- 'nyt' vs 'bbc' — both 3 chars, would get high similarity — wrong
- 'fox' vs 'vox' — one letter apart — completely different outlets
- 'ap' vs 'nbc' — would incorrectly merge

So we hardcode: for each source, list every variant we expect.
More specific patterns come first. Exceptions (fox 13, google discover) come before their general rule.

In [11]:
# Each entry: (regex pattern, standard name)
# \b = word boundary — e.g. \bcnn\b matches 'cnn' but not 'xcnn' or 'cnn2'

ALIAS_MAP = [
    # Exceptions — must come before their general pattern
    (r'\bfox\s*1[0-9]\b',                   'Fox 13'),           # fox13, fox10, fox 13 — local affiliates
    (r'\bgoogle\s+discover\b',               'Google Discover'),  # different from Google News

    # New York Times
    (r'\bthe\s+new\s+york\s+times?\b',       'New York Times'),
    (r'\bnew\s+york\s+times?\b',             'New York Times'),
    (r'\bnytimes?\b',                        'New York Times'),
    (r'\bny\s+times?\b',                     'New York Times'),
    (r'\bnyt\b',                             'New York Times'),

    # CNN
    (r'\bcnn\s*news?\b',                     'CNN'),
    (r'\bcnn\b',                             'CNN'),

    # Sky News
    (r'\bsky\s*news\b',                      'Sky News'),
    (r'\bskynews\b',                         'Sky News'),

    # Fox News — after fox13 exception above
    # \bfox\b(?!\s*\d) = 'fox' not followed by a number (fox13 already caught)
    (r'\bfox\s+news\b',                      'Fox News'),
    (r'\bfoxnews\b',                         'Fox News'),
    (r'\blivenow\s+from\s+fox\b',            'Fox News'),
    (r'\bfox\b(?!\s*\d)',                    'Fox News'),

    # BBC
    (r'\bbbcnews\b',                         'BBC News'),
    (r'\bbbc\s+world\s+news\b',              'BBC News'),
    (r'\bbbc\s+world\b',                     'BBC News'),
    (r'\bbbc(?:\s+news)?\b',                 'BBC News'),

    # Washington Post
    (r'\bthe\s+washington\s+post\b',         'Washington Post'),
    (r'\bwashington\s+post\b',               'Washington Post'),
    (r'\bwapo\b',                            'Washington Post'),
    (r'\bwashpost\b',                        'Washington Post'),
    (r'\bwashingtonpost\b',                  'Washington Post'),
    (r'\btwp\b',                             'Washington Post'),

    # MSNBC
    (r'\bmsnbc\b',                           'MSNBC'),

    # NBC
    (r'\bnbc\s+nightly\s+news\b',            'NBC News'),
    (r'\bnbc\s+washington\b',                'NBC News'),
    (r'\bnbc(?:\s+news)?\b',                 'NBC News'),

    # ABC
    (r'\babc\s+world\s+news\b',              'ABC News'),
    (r'\bgood\s+morning\s+america\b',        'ABC News'),  # ABC flagship show
    (r'\babc\s*\d+\s*(?:news)?\b',           'ABC News'),  # abc7, abc7news
    (r'\babc(?:\s+news)?\b',                 'ABC News'),

    # CBS
    (r'\bcbs\s+evening\s+news\b',            'CBS News'),
    (r'\b60\s+minutes\b',                    'CBS News'),  # CBS flagship show
    (r'\bcbs(?:\s+news)?\b',                 'CBS News'),

    # NPR — including long forms people write
    # 'npr (national public radio)', 'national public radio', 'npr morning news'
    (r'\bnpr\b',                             'NPR'),
    (r'\bnational\s+public\s+radio\b',       'NPR'),
    (r'\bnpr\s+morning\b',                   'NPR'),

    # PBS
    (r'\bpbs(?:\s+news(?:hour)?)?\b',        'PBS NewsHour'),

    # Associated Press
    # bare 'ap' is only 2 letters — small false-match risk but AP is high-frequency
    (r'\bassociated\s+press\b',              'Associated Press'),
    (r'\bapnews\b',                          'Associated Press'),
    (r'\bap\s+news\b',                       'Associated Press'),
    (r'\bap\b',                              'Associated Press'),

    # Reuters
    (r'\breuters\b',                         'Reuters'),

    # Wall Street Journal
    (r'\bwall\s+street\s+journal\b',         'Wall Street Journal'),
    (r'\bthe\s+wall\s+street\s+journal\b',   'Wall Street Journal'),
    (r'\bwsj\b',                             'Wall Street Journal'),

    # Al Jazeera — bare '\bal\b' NOT included (too many false matches: Alabama, etc.)
    (r'\bal\s+jazeera\b',                    'Al Jazeera'),
    (r'\baljazeera\b',                       'Al Jazeera'),

    # Yahoo News
    (r'\byahoo(?:\s+news)?\b',               'Yahoo News'),

    # Google News (not Discover — already caught above)
    (r'\bgoogle(?:\s+news)?\b',              'Google News'),

    # Atlanta Journal-Constitution
    (r'\batlanta\s+journal[\s\-]+constitution\b', 'Atlanta Journal-Constitution'),
    (r'\batlanta\s+journal\b',               'Atlanta Journal-Constitution'),
    (r'\bajc\b',                             'Atlanta Journal-Constitution'),

    # Social media platforms
    (r'\bfacebook\b',                        'Facebook'),
    (r'\bfb\b',                              'Facebook'),
    (r'\binstagram\b',                       'Instagram'),
    (r'\big\b(?!n)',                         'Instagram'),   # 'ig' but not 'ign'
    (r'\btiktok\b',                          'TikTok'),
    (r'\btwitter\b',                         'Twitter/X'),
    (r'\bx\.com\b',                          'Twitter/X'),
    (r'\byoutube\b',                         'YouTube'),
    (r'\byt\b',                              'YouTube'),
    (r'\breddit\b',                          'Reddit'),
    (r'\bsnapchat\b',                        'Snapchat'),
    (r'\bdiscord\b',                         'Discord'),
    (r'\bpinterest\b',                       'Pinterest'),
    (r'\blinkedin\b',                        'LinkedIn'),
    (r'\bthreads\b',                         'Threads'),
    (r'\bbluesky\b',                         'Bluesky'),
    (r'\bmastodon\b',                        'Mastodon'),
    (r'\btwitch\b',                          'Twitch'),

    # Other outlets
    (r'\busa\s+today\b',                     'USA Today'),
    (r'\bla\s+times\b',                      'Los Angeles Times'),
    (r'\blos\s+angeles\s+times\b',           'Los Angeles Times'),
    (r'\blatimes\b',                         'Los Angeles Times'),
    (r'\bboston\s+globe\b',                  'Boston Globe'),
    (r'\bnewsweek\b',                        'Newsweek'),
    (r'\btime(?:\s+magazine)?\b',            'TIME Magazine'),
    (r'\bpolitico\b',                        'Politico'),
    (r'\bbloomberg\b',                       'Bloomberg'),
    (r'\bcnbc\b',                            'CNBC'),
    (r'\bforbes\b',                          'Forbes'),
    (r'\beconomist\b',                       'The Economist'),
    (r'\bfinancial\s+times\b',               'Financial Times'),
    (r'\bft\b',                              'Financial Times'),
    (r'\bvox\b',                             'Vox'),
    (r'\bthe\s+atlantic\b',                  'The Atlantic'),
    (r'\batlantic\b',                        'The Atlantic'),
    (r'\bnew\s+yorker\b',                    'The New Yorker'),
    (r'\bguardian\b',                        'The Guardian'),
    (r'\btheguardian\b',                     'The Guardian'),
    (r'\baxios\b',                           'Axios'),
    (r'\bthe\s+hill\b',                      'The Hill'),
    (r'\bhuffpost\b',                        'HuffPost'),
    (r'\bhuffington\s+post\b',               'HuffPost'),
    (r'\bbuzzfeed\b',                        'BuzzFeed'),
    (r'\bespn\b',                            'ESPN'),
    (r'\bbreitbart\b',                       'Breitbart'),
    (r'\bdaily\s+wire\b',                    'Daily Wire'),
    (r'\bthe\s+daily\s+wire\b',              'Daily Wire'),
    (r'\bthe\s+blaze\b',                     'The Blaze'),
    (r'\bdaily\s+mail\b',                    'Daily Mail'),
    (r'\bnypost\b',                          'New York Post'),
    (r'\bnew\s+york\s+post\b',               'New York Post'),
    (r'\bny\s+post\b',                       'New York Post'),
    (r'\bmsn\b',                             'MSN News'),
    (r'\bapple\s+news\b',                    'Apple News'),
    (r'\bflipboard\b',                       'Flipboard'),
    (r'\bground\.news\b',                    'Ground News'),
    (r'\bground\s+news\b',                   'Ground News'),
    (r'\bnewsmax\b',                         'Newsmax'),
    (r'\bnewsnation\b',                      'NewsNation'),
    (r'\bdw\b',                              'DW (Deutsche Welle)'),
    (r'\bdeutsche\s+welle\b',                'DW (Deutsche Welle)'),
    (r'\bprobublica\b',                      'ProPublica'),
    (r'\bpropublica\b',                      'ProPublica'),
    (r'\btmz\b',                             'TMZ'),
    (r'\bslate\b',                           'Slate'),
    (r'\bvice\b',                            'Vice'),
    (r'\bsalon\b',                           'Salon'),
    (r'\bzerohedge\b',                       'ZeroHedge'),
    (r'\bmarketwatch\b',                     'MarketWatch'),
    (r'\btelemundo\b',                       'Telemundo'),
    (r'\bunivision\b',                       'Univision'),
    (r'\bcbc\b',                             'CBC'),
    (r'\bdrudge\s+report\b',                 'Drudge Report'),
    (r'\bdrudge\b',                          'Drudge Report'),
    (r'\breal\s+clear\s+politics\b',         'RealClearPolitics'),
    (r'\bthe\s+daily\s+show\b',              'The Daily Show'),
    (r'\bdaily\s+show\b',                    'The Daily Show'),
    (r'\bbreaking\s+points\b',               'Breaking Points'),
    (r'\bdemocracy\s+now\b',                 'Democracy Now'),
    (r'\bpod\s+save\s+america\b',            'Pod Save America'),
    (r'\bmeidas\s*touch\b',                  'MeidasTouch'),
    (r'\bthe\s+young\s+turks\b',             'TYT'),
    (r'\btyt\b',                             'TYT'),
    (r'\bgreenbrier\s+news\b',               'Greenbrier News'),
    (r'\bnews\s*break\b',                    'NewsBreak'),
    (r'\bthehill\b',                         'The Hill'),
    (r'\bpolitifact\b',                      'PolitiFact'),
    (r'\bsnopes\b',                          'Snopes'),
    (r'\b1440\b',                            '1440 Daily Digest'),
    (r'\bunder\s*the\s*desk\s*news\b',       'Under the Desk News'),
    (r'\bunderthedesknews\b',                'Under the Desk News'),
]


def unify(s):
    for pattern, standard_name in ALIAS_MAP:
        if re.search(pattern, s):
            return standard_name
    return s.title()


unified_series = all_extracted.apply(unify)

unified_freq = unified_series.value_counts().reset_index()
unified_freq.columns = ['source', 'count']

unique_before = all_extracted.nunique()
unique_after = unified_series.nunique()

print(f"Unique strings before merging: {unique_before}")
print(f"Unique strings after merging:  {unique_after}")
print(f"Reduced by: {unique_before - unique_after} ({round((unique_before - unique_after) / unique_before * 100, 1)}%)")
print()

top_unified = unified_freq[unified_freq['count'] >= 3].copy()
print(f"Sources mentioned 3+ times: {len(top_unified)}")
print()
with pd.option_context('display.max_rows', 300, 'display.max_colwidth', 60):
    print(top_unified.to_string())

Unique strings before merging: 2332
Unique strings after merging:  1997
Reduced by: 335 (14.4%)

Sources mentioned 3+ times: 223

                           source  count
0                             CNN    651
1                        Fox News    317
2                  New York Times    295
3                        ABC News    219
4                        NBC News    207
5                        BBC News    206
6                           MSNBC    199
7                             NPR    194
8                        CBS News    144
9                 Washington Post    123
10                    Google News     79
11            Wall Street Journal     77
12               Associated Press     72
13                        Reuters     65
14                     Yahoo News     58
15                           CNBC     48
16                   The Guardian     47
17                     Local News     40
18                      USA Today     37
19                     Al Jazeera     35
20       

# Step 7: Label each source — MM, SM, MM-in-SM, or NN

Four categories:
- MM = Mainstream Media: traditional news outlets, written in a traditional media column
- MM-in-SM = Mainstream Media accessed via social media: a news source written in a social media column (e.g. someone typed 'CNN' in the YouTube column)
- SM = Social Media platform: Facebook, TikTok, YouTube, etc.
- NN = Non-News: entertainment, noise, n/a

Priority: NN first, then SM, then MM vs MM-in-SM based on which survey column it came from.

In [12]:
SM_PATTERNS = [
    r'\bfacebook\b', r'\binstagram\b', r'\btiktok\b',
    r'\btwitter\b',  r'\byoutube\b',   r'\breddit\b',
    r'\bsnapchat\b', r'\bdiscord\b',   r'\bpinterest\b',
    r'\blinkedin\b', r'\bthreads\b',   r'\bbluesky\b',
    r'\bmastodon\b', r'\bfyp\b',       r'\bx\.com\b',
    r'\btwitch\b',
]

NN_PHRASES = [
    'bands', 'people i follow', 'shows up', 'for you', 'whatever',
    'friends', 'feed', 'algorithm', 'random', 'not sure', 'timeline',
    'pop up', "don't follow", "don't know", 'no particular', 'stumble',
    'entertainment', 'comedy', 'gaming', 'music', 'podcast',
    'movie', 'film', 'anime', 'manga', 'video game',
    'reels', 'meme', 'content creators', 'various content',
]

NN_EXACT = [r'\bna\b', r'\bn/a\b', r'\bnone\b', r'\bnothing\b', r'\bn\b']


def label(original, unified, origin):
    # origin = 'SM_col' or 'MM_col' (set in step 2)
    lo = original.lower()
    lu = unified.lower()

    # Long unresolved sentences are noise
    if len(lo) > 40:
        return 'NN'

    # Too many words — probably a sentence fragment, not a source name
    if len(lo.split()) > 5:
        return 'NN'

    if any(phrase in lo for phrase in NN_PHRASES):
        return 'NN'

    if any(re.search(kw, lo) for kw in NN_EXACT):
        return 'NN'

    # Pure number (e.g. leftover frequency noise)
    if re.fullmatch(r'\d+', lo):
        return 'NN'

    # Social media platform name
    if any(re.search(p, lo) or re.search(p, lu) for p in SM_PATTERNS):
        return 'SM'

    # News source written in a social media column = MM-in-SM
    if origin == 'SM_col':
        return 'MM-in-SM'

    # Everything else = regular Mainstream Media
    return 'MM'


# Build source table — one row per unique (original_cleaned, survey_col) combo
source_table = exploded_df.copy()
source_table['unified_source'] = unified_series.values
source_table['category'] = source_table.apply(
    lambda row: label(row['cleaned'], row['unified_source'], row['origin']), axis=1
)

# Frequency of each cleaned string
freq_map = all_cleaned.value_counts().to_dict()
source_table['freq'] = source_table['cleaned'].map(freq_map)
source_table = source_table.sort_values('freq', ascending=False).reset_index(drop=True)

print("Category counts:")
print(source_table['category'].value_counts())
print()
print("Top 40:")
with pd.option_context('display.max_rows', 40, 'display.max_colwidth', 50):
    print(source_table[['cleaned', 'unified_source', 'category', 'freq']].head(40).to_string())

Category counts:
category
MM          3622
MM-in-SM    2247
NN           411
SM            75
Name: count, dtype: int64

Top 40:
   cleaned unified_source  category  freq
0      cnn            CNN        MM   623
1      cnn            CNN        MM   623
2      cnn            CNN  MM-in-SM   623
3      cnn            CNN        MM   623
4      cnn            CNN        MM   623
5      cnn            CNN        MM   623
6      cnn            CNN        MM   623
7      cnn            CNN        MM   623
8      cnn            CNN        MM   623
9      cnn            CNN  MM-in-SM   623
10     cnn            CNN        MM   623
11     cnn            CNN        MM   623
12     cnn            CNN        MM   623
13     cnn            CNN        MM   623
14     cnn            CNN  MM-in-SM   623
15     cnn            CNN        MM   623
16     cnn            CNN        MM   623
17     cnn            CNN        MM   623
18     cnn            CNN        MM   623
19     cnn            CNN      

# Step 7b: Political leaning from Media Bias Fact Check

Each unique unified source gets a political leaning label based on mediabiasfactcheck.com.
Scale: Left / Left-Center / Center / Right-Center / Right / Unknown

Sources not in the list are labeled Unknown — they are usually local stations, minor blogs, or people (podcasters, commentators).

In [13]:
# Political leaning per source — from mediabiasfactcheck.com
# Left = strongly left leaning   Left-Center = slightly left
# Center = balanced/factual      Right-Center = slightly right   Right = strongly right
POLITICAL_LEANING = {
    'CNN':                        'Left-Center',
    'Fox News':                   'Right',
    'MSNBC':                      'Left',
    'NPR':                        'Left-Center',
    'BBC News':                   'Center',
    'New York Times':             'Left-Center',
    'Washington Post':            'Left-Center',
    'Wall Street Journal':        'Right-Center',
    'NBC News':                   'Left-Center',
    'ABC News':                   'Left-Center',
    'CBS News':                   'Left-Center',
    'PBS NewsHour':               'Left-Center',
    'Associated Press':           'Center',
    'Reuters':                    'Center',
    'USA Today':                  'Left-Center',
    'Los Angeles Times':          'Left-Center',
    'Boston Globe':               'Left-Center',
    'New York Post':              'Right',
    'Newsmax':                    'Right',
    'Breitbart':                  'Right',
    'Daily Wire':                 'Right',
    'The Blaze':                  'Right',
    'Daily Mail':                 'Right-Center',
    'The Hill':                   'Center',
    'HuffPost':                   'Left',
    'Vox':                        'Left',
    'The Atlantic':               'Left-Center',
    'The New Yorker':             'Left-Center',
    'The Guardian':               'Left-Center',
    'Axios':                      'Center',
    'Bloomberg':                  'Center',
    'CNBC':                       'Center',
    'Forbes':                     'Right-Center',
    'The Economist':              'Center',
    'Financial Times':            'Center',
    'Politico':                   'Center',
    'Newsweek':                   'Left-Center',
    'TIME Magazine':              'Left-Center',
    'Al Jazeera':                 'Left-Center',
    'BuzzFeed':                   'Left',
    'Slate':                      'Left',
    'Salon':                      'Left',
    'Vice':                       'Left',
    'Vox':                        'Left',
    'Democracy Now':              'Left',
    'TYT':                        'Left',
    'MSN News':                   'Center',
    'Yahoo News':                 'Center',
    'Google News':                'Center',
    'Apple News':                 'Center',
    'Sky News':                   'Right-Center',
    'DW (Deutsche Welle)':        'Center',
    'CBC':                        'Left-Center',
    'Telemundo':                  'Center',
    'Univision':                  'Center',
    'ProPublica':                 'Left-Center',
    'ZeroHedge':                  'Right',
    'RealClearPolitics':          'Right-Center',
    'Drudge Report':              'Right',
    'Atlanta Journal-Constitution': 'Left-Center',
    'NewsNation':                 'Center',
    'Breaking Points':            'Center',
    'Pod Save America':           'Left',
    'MeidasTouch':                'Left',
    'The Daily Show':             'Left',
    'Greenbrier News':            'Unknown',
    'Ground News':                'Center',
    'NewsBreak':                  'Center',
    'Under the Desk News':        'Left',
    '1440 Daily Digest':          'Center',
    'MarketWatch':                'Center',
    'TMZ':                        'Unknown',
    'Snopes':                     'Left-Center',
    'PolitiFact':                 'Left-Center',
    'ESPN':                       'Unknown',
    'Fox 13':                     'Unknown',
    'Google Discover':            'Center',
    # Social media — label as N/A (not a news source itself)
    'Facebook':    'N/A',
    'Instagram':   'N/A',
    'TikTok':      'N/A',
    'Twitter/X':   'N/A',
    'YouTube':     'N/A',
    'Reddit':      'N/A',
    'Snapchat':    'N/A',
    'Discord':     'N/A',
    'Pinterest':   'N/A',
    'LinkedIn':    'N/A',
    'Threads':     'N/A',
    'Bluesky':     'N/A',
    'Mastodon':    'N/A',
    'Twitch':      'N/A',
}

source_table['political_leaning'] = source_table['unified_source'].map(POLITICAL_LEANING).fillna('Unknown')

print("Political leaning distribution:")
print(source_table['political_leaning'].value_counts())

Political leaning distribution:
political_leaning
Unknown         2669
Left-Center     2042
Center           702
Right            407
Left             330
Right-Center     112
N/A               93
Name: count, dtype: int64


# Step 8: All unique sources with unified name, category, leaning, and frequency

This is the main reference table. Shows every unique cleaned string, what it was mapped to,
its category, political leaning, and how many times it appeared. Good for spot-checking.

In [14]:
# One row per unique unified source — aggregate frequency across all variants
audit_table = (
    source_table
    .groupby(['unified_source', 'category', 'political_leaning'])['freq']
    .sum()
    .reset_index()
    .rename(columns={'freq': 'total_mentions'})
    .sort_values('total_mentions', ascending=False)
    .reset_index(drop=True)
)

print(f"Unique unified sources: {len(audit_table)}")
print()
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', 50):
    print(audit_table.to_string())

Unique unified sources: 2252

                                                                                                      unified_source  category political_leaning  total_mentions
0                                                                                                                CNN        MM       Left-Center          245536
1                                                                                                                CNN  MM-in-SM       Left-Center          142699
2                                                                                                           Fox News        MM             Right           31155
3                                                                                                              MSNBC        MM              Left           22094
4                                                                                                                NPR        MM       Left-Center           19836
5   

# Step 9: Per-person counts by source type and political leaning

For each participant, count how many unique sources they mentioned of each type:
- mm_count = unique Mainstream Media sources
- mm_in_sm_count = unique MM sources accessed via social media columns
- sm_count = unique Social Media platforms
- Then counts per political leaning: left_count, left_center_count, center_count, right_center_count, right_count

These are counts of unique sources (not mentions) — so if someone mentioned CNN three times, it counts as 1.

In [15]:
# Build lookup from cleaned string → (unified, category, political_leaning)
lookup = (
    source_table
    .drop_duplicates(subset='cleaned')
    .set_index('cleaned')[['unified_source', 'category', 'political_leaning']]
    .to_dict('index')
)

records = []
for idx, row in df.iterrows():
    pid = row.get('PROLIFIC_PID', idx)

    # Collect all (unified_source, category, political_leaning) tuples for this person
    found = []
    for col in source_cols:
        cell = str(row[col])
        if cell in ('nan', ''):
            continue
        origin = 'SM_col' if col in SM_SURVEY_COLS else 'MM_col'
        for seg in split_sources(cell):
            norm = clean(seg)
            if not norm:
                continue
            extracted = pull_source_from_sentence(norm, known_names)
            if len(extracted) >= 25:
                extracted = extract_keyword_pattern(extracted)
            uni = unify(extracted)
            cat = label(extracted, uni, origin)
            pol = POLITICAL_LEANING.get(uni, 'Unknown')
            found.append((uni, cat, pol))

    # Count unique unified sources per category
    mm_sources      = {u for u, c, p in found if c == 'MM'}
    mm_in_sm_sources = {u for u, c, p in found if c == 'MM-in-SM'}
    sm_sources      = {u for u, c, p in found if c == 'SM'}

    # Count unique unified sources per political leaning (only MM and MM-in-SM)
    news_only = {(u, p) for u, c, p in found if c in ('MM', 'MM-in-SM')}
    leaning_counts = {}
    for leaning in ['Left', 'Left-Center', 'Center', 'Right-Center', 'Right', 'Unknown']:
        leaning_counts[leaning.lower().replace('-', '_') + '_count'] = sum(
            1 for u, p in news_only if p == leaning
        )

    record = {
        'PROLIFIC_PID':      pid,
        'mm_count':          len(mm_sources),
        'mm_in_sm_count':    len(mm_in_sm_sources),
        'sm_count':          len(sm_sources),
        'total_news_count':  len(mm_sources | mm_in_sm_sources),
    }
    record.update(leaning_counts)
    record['sources_list'] = sorted(mm_sources | mm_in_sm_sources)

    records.append(record)

diversity_df = pd.DataFrame(records)
print(diversity_df.drop(columns='sources_list').describe().round(2))
print()
with pd.option_context('display.max_rows', 10, 'display.max_colwidth', 60):
    print(diversity_df.drop(columns='sources_list').head(10).to_string())

       mm_count  mm_in_sm_count  sm_count  total_news_count  left_count  \
count    928.00          928.00    928.00            928.00      928.00   
mean       3.55            2.14      0.09              5.19        0.27   
std        3.21            2.90      0.30              4.33        0.55   
min        0.00            0.00      0.00              0.00        0.00   
25%        1.00            0.00      0.00              2.00        0.00   
50%        3.00            1.00      0.00              4.00        0.00   
75%        5.00            3.00      0.00              7.00        0.00   
max       22.00           32.00      2.00             32.00        4.00   

       left_center_count  center_count  right_center_count  right_count  \
count             928.00        928.00              928.00       928.00   
mean                1.51          0.59                0.09         0.29   
std                 1.63          0.89                0.31         0.54   
min                 0.00

# Step 10: Export to Excel

In [16]:
excel_filename = 'source_extraction_pipeline.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:

    # Sheet 1: audit table — all unique sources with unified name, category, leaning, frequency
    audit_table.to_excel(writer, index=False, sheet_name='Source Audit')
    ws = writer.sheets['Source Audit']
    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for col_idx in range(1, len(audit_table.columns) + 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = 35

    # Sheet 2: full source table including all raw variants
    source_table.to_excel(writer, index=False, sheet_name='Source Dictionary')
    ws2 = writer.sheets['Source Dictionary']
    for row in ws2.iter_rows():
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for col_idx in range(1, len(source_table.columns) + 1):
        ws2.column_dimensions[get_column_letter(col_idx)].width = 30

    # Sheet 3: per-person counts
    diversity_export = diversity_df.drop(columns='sources_list')
    diversity_export.to_excel(writer, index=False, sheet_name='Per-Person Counts')
    ws3 = writer.sheets['Per-Person Counts']
    for row in ws3.iter_rows():
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top')
    for col_idx in range(1, len(diversity_export.columns) + 1):
        ws3.column_dimensions[get_column_letter(col_idx)].width = 20

print(f"Saved: {excel_filename}")
print(f"  Source Audit:       {len(audit_table)} rows")
print(f"  Source Dictionary:  {len(source_table)} rows")
print(f"  Per-Person Counts:  {len(diversity_export)} rows")

Saved: source_extraction_pipeline.xlsx
  Source Audit:       2252 rows
  Source Dictionary:  6355 rows
  Per-Person Counts:  928 rows


# Summary and known limitations

What works well:
- Common sources (CNN, Fox News, NYT, BBC, etc.) are correctly merged
- MM-in-SM subcategory tracks when news sources are accessed through social media platforms
- Sources extracted from sentences via anchor scan, 'like' pattern, and 'from' pattern
- NPR (national public radio) and Atlanta Journal-Constitution explicitly unified
- Political leaning from Media Bias Fact Check for all major sources

Known trade-offs:
- Sources mentioned by only 1–2 people may not be perfectly unified
- Bare 'ap' occasionally catches non-AP uses — acceptable given AP's frequency
- Local stations (WFLA, KCCI, etc.) get leaning = Unknown
- Commentator names (Joe Rogan, Tucker Carlson) are labeled Unknown leaning and MM

To improve:
- Add more entries to ALIAS_MAP as you spot new variants in the trace table
- Add political leanings for local stations if needed
- Check NN items in the Source Dictionary — some may need recategorizing